<a href="https://colab.research.google.com/github/tarun161/AILearning/blob/llm-conversations/GoogleADk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text and Chat Completion

In [ ]:
import os
from google import genai
from google.colab import userdata

gemini_api_key = userdata.get('GEMINI_API_KEY')


client = genai.Client(api_key=gemini_api_key)

# 1. Single Text Generation (Stateless)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain dependency injection in two sentences."
)
print("Single Generation:", response.text)



In [ ]:
# 2. Chat Completions (Stateful)
chat = client.chats.create(model="gemini-2.5-flash")
chat.send_message("I am building a microservices architecture.")
response2 = chat.send_message("What did I just tell you I was building?")
print(response2)
print("Chat Response:", response2.text)

In [ ]:
#Model Thinking
# More on Thinking Config (https://ai.google.dev/gemini-api/docs/thinking#set-budget)
from google.genai import types

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Provide a list of 3 famous physicists and their key contributions",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=1024)
        # Turn off thinking:
        # thinking_config=types.ThinkingConfig(thinking_budget=0)
        # Turn on dynamic thinking:
        # thinking_config=types.ThinkingConfig(thinking_budget=-1)
    ),
)

print(response.text)

In [ ]:
# System Instructions

reponse = client.models.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a cat. Your name is Neko."),
    contents="Hello there whats your name"
)
print(response)

In [ ]:
# temperature settings

response = client.model.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a cat. Your name is Neko.",
        temperature=0.1),
    contents="Hello there whats your name"
)

In [ ]:
# Streaming responses

response = client.models.generate_content_stream(
    model="gemini-2.5-flash",
    contents=["Explain how AI works"]
)
for chunk in response:
    print(chunk.text, end="")

In [ ]:
# Multi-turn conversations

chat = client.chats.create(model="gemini-3.5-flash")

response = chat.send_message("I have 2 dogs in my house.")
print(response.text)

response = chat.send_message("How many paws are in my house?")
print(response.text)

for message in chat.get_history():
    print(f'role - {message.role}',end=": ")
    print(message.parts[0].text)

# Cient as OPENAI


In [ ]:
!pip install openai

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

# Retrieve the OpenAI API key from Colab secrets
openai_api_key = userdata.get('OPENAI_KEY')

# Initialize the OpenAI client
client = OpenAI(api_key=openai_api_key)

# 1. Single Text Generation (Stateless)
response = client.chat.completions.create(
    model="gpt-4o", # You can also use "gpt-4o-mini" or "gpt-4-turbo"
    messages=[
        {"role": "user", "content": "Explain dependency injection in two sentences."}
    ]
)

# Extract and print the response text
print("Single Generation:", response.choices[0].message.content)

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

# Initialize the OpenAI client
openai_api_key = userdata.get('OPENAI_KEY')
client = OpenAI(api_key=openai_api_key)

# 2. Chat Completions (Stateful)
# In OpenAI, state is managed by keeping the history in a list
messages = [
    {"role": "user", "content": "I am building a microservices architecture."}
]

# Send the first message (simulated here by adding it to the list)
# Now we add the second message to the history
messages.append({"role": "user", "content": "What did I just tell you I was building?"})

response2 = client.chat.completions.create(
    model="gpt-4o",
    messages=messages
)

print("Full Response Object:", response2)
print("Chat Response:", response2.choices[0].message.content)

In [ ]:
# Streaming with open AI


chat_history = []

# Step 1: Add user prompt
prompt = "I am building a microservices architecture. Suggest a good stack for it."
chat_history.append({"role": "user", "content": prompt})

# Step 2: Request stream
response_stream = client.chat.completions.create(
    model="gpt-4o",
    messages=chat_history,
    stream=True # Activates streaming
)

# Step 3: Print chunks as they arrive and accumulate the final text
full_reply = ""
print("Chat Response: ", end="")

for chunk in response_stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content,end="")

